#  Pothole Detection (10,000+ Images)  YOLOv8 Training on GPU

This notebook trains **YOLOv8s** on a **10,000+ image pothole dataset** using free GPU acceleration on Google Colab / Kaggle.

### Pipeline Overview:
1. **Environment:** Verify NVIDIA GPU (Tesla T4 / V100 / A100)
2. **Dataset:** Download 10,000+ annotated pothole road images (Roboflow / RDD2022)
3. **Training:** Train YOLOv8s with CIoU + DFL + BCE loss, Mosaic + HSV augmentation for 50-100 epochs
4. **Evaluation:** Generate precision-recall curves, confusion matrix, mAP@0.5 and mAP@0.5:0.95
5. **Export:** Download `best.pt` directly to place into `backend/weights/pothole_det.pt`

In [ ]:
# Step 1: Verify GPU Acceleration
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print(" Please go to Runtime -> Change runtime type -> Select T4 GPU!")

In [ ]:
# Step 2: Install Ultralytics and dependencies
!pip install -q ultralytics roboflow albumentations

In [ ]:
# Step 3: Download the 13,767-image Pothole Dataset
# Source: Roboflow Universe Project SSAYL (Potholes Detection - 13,767 images)
import os

# Download the complete 13,767-image pothole dataset in YOLOv8 format
!curl -L "https://universe.roboflow.com/ds/7t6pG4u22U?key=P1eB2g0N1K" > roboflow_potholes.zip; unzip -q roboflow_potholes.zip -d datasets/potholes_10k; rm roboflow_potholes.zip

# Verify counts
import glob
train_imgs = glob.glob('datasets/potholes_10k/train/images/*')
val_imgs = glob.glob('datasets/potholes_10k/valid/images/*')
test_imgs = glob.glob('datasets/potholes_10k/test/images/*')
print(f"Total Images Downloaded:")
print(f"  Train: {len(train_imgs)}")
print(f"  Valid: {len(val_imgs)}")
print(f"  Test:  {len(test_imgs)}")
print(f"  TOTAL: {len(train_imgs) + len(val_imgs) + len(test_imgs)} images")

In [ ]:
# Step 4: Train YOLOv8s for 50 Epochs
from ultralytics import YOLO

# Load pre-trained YOLOv8s backbone for transfer learning
model = YOLO('yolov8s.pt')

# Train with GPU acceleration
results = model.train(
    data='datasets/potholes_10k/data.yaml',
    epochs=50,
    imgsz=640,
    batch=32,            # 32 images per batch on T4 GPU (16GB VRAM)
    device=0,            # Use GPU
    patience=15,         # Early stopping if mAP doesn't improve for 15 epochs
    save=True,
    project='pothole_training_10k',
    name='yolov8s_10k_run',
    mosaic=1.0,          # Combine 4 images into 1 for scale invariance
    hsv_h=0.015,         # Color jitter (handles varying sunlight)
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5           # Random horizontal flip
)

In [ ]:
# Step 5: Evaluate Model on Held-out Test Split
test_metrics = model.val(data='datasets/potholes_10k/data.yaml', split='test')
print("="*50)
print("FINAL TEST METRICS (10,000+ Image Trained Model):")
print(f"  Precision:    {test_metrics.box.mp:.4f}")
print(f"  Recall:       {test_metrics.box.mr:.4f}")
print(f"  mAP@0.5:      {test_metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {test_metrics.box.map:.4f}")
print("="*50)

In [ ]:
# Step 6: Display Training Curves & Confusion Matrix
from IPython.display import Image, display
import os

run_dir = 'pothole_training_10k/yolov8s_10k_run'
for plot in ['results.png', 'confusion_matrix.png', 'BoxPR_curve.png', 'BoxF1_curve.png']:
    p = os.path.join(run_dir, plot)
    if os.path.exists(p):
        print(f"Displaying {plot}:")
        display(Image(filename=p))

In [ ]:
# Step 7: Download your trained weights directly to your computer
from google.colab import files
best_weights = 'pothole_training_10k/yolov8s_10k_run/weights/best.pt'
print(f"Downloading {best_weights} ...")
files.download(best_weights)
print("Replace backend/weights/pothole_det.pt with this downloaded best.pt!")